In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pickle
import json
from IDS import IntrusionDetector
from simple_rules import AttackPlausibilityChecker

# --- 1. Load preprocessed data and model ---
X_train = np.load('results/X_train.npy')
X_test = np.load('results/X_test.npy')
y_train = np.load('results/y_train.npy')
y_test = np.load('results/y_test.npy')
test_original_labels = np.load('results/test_original_labels.npy', allow_pickle=True)

with open('results/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open('results/feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)
with open('nslkdd_features.json', 'r') as f:
    features_meta = json.load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = IntrusionDetector(input_dim=X_train.shape[1]).to(device)
model.load_state_dict(torch.load('ids_model.pth', map_location=device))
model.eval()

# --- 2. Build modifiability mask ---
# Map original feature names to their modifiability
modifiability_map = {}
for feat in features_meta:
    modifiability_map[feat['name']] = feat['modifiability']

# Create mask for HIGHLY_MODIFIABLE features (in one-hot encoded space)
highly_modifiable_mask = np.zeros(len(feature_names), dtype=bool)
partially_modifiable_mask = np.zeros(len(feature_names), dtype=bool)

for i, fname in enumerate(feature_names):
    # For one-hot encoded features, get the base name (before '_')
    base_name = fname
    for orig_name in modifiability_map:
        if fname == orig_name or fname.startswith(orig_name + '_'):
            mod_level = modifiability_map[orig_name]
            if mod_level == 'HIGHLY_MODIFIABLE':
                highly_modifiable_mask[i] = True
            elif mod_level == 'PARTIALLY_MODIFIABLE':
                partially_modifiable_mask[i] = True
            break

# Combined mask: highly + partially modifiable
combined_modifiable_mask = highly_modifiable_mask | partially_modifiable_mask

print(f"Total features: {len(feature_names)}")
print(f"Highly modifiable: {highly_modifiable_mask.sum()}")
print(f"Partially modifiable: {partially_modifiable_mask.sum()}")
print(f"Combined modifiable: {combined_modifiable_mask.sum()}")

# --- 3. Compute valid feature ranges from training data (standardized) ---
feature_min = X_train.min(axis=0)
feature_max = X_train.max(axis=0)

# --- 4. Identify correctly classified attack samples ---
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
with torch.no_grad():
    preds = model(X_test_t).cpu().numpy().flatten()
    pred_labels = (preds >= 0.5).astype(int)

# True positives: actual attacks that IDS correctly identifies as attacks
true_positive_mask = (y_test == 1) & (pred_labels == 1)
tp_indices = np.where(true_positive_mask)[0]
print(f"\nCorrectly classified attacks (true positives): {len(tp_indices)}")

# Determine attack type for each test sample (for plausibility checking)
attack_type_map = {}
for feat in features_meta:
    if feat['name'] == 'label':
        for label, category in feat['flag_values'].items():
            if 'DoS' in category:
                attack_type_map[label] = 'DoS'
            elif 'Probe' in category:
                attack_type_map[label] = 'Probe'
            elif 'R2L' in category:
                attack_type_map[label] = 'R2L'
            elif 'U2R' in category:
                attack_type_map[label] = 'U2R'

# --- 5. PGD Attack ---
def pgd_attack(model, x_orig, modifiable_mask, epsilon, alpha=0.01,
               num_iter=40, feat_min=None, feat_max=None):
    """
    Constrained PGD attack to evade IDS.
    Goal: make model output < 0.5 (classify as normal).
    Only perturbs features indicated by modifiable_mask.
    """
    x_adv = torch.tensor(x_orig.copy(), dtype=torch.float32, device=device).unsqueeze(0)
    x_orig_t = torch.tensor(x_orig.copy(), dtype=torch.float32, device=device).unsqueeze(0)

    mask_t = torch.tensor(modifiable_mask, dtype=torch.float32, device=device)
    feat_min_t = torch.tensor(feat_min, dtype=torch.float32, device=device)
    feat_max_t = torch.tensor(feat_max, dtype=torch.float32, device=device)

    for _ in range(num_iter):
        x_adv.requires_grad_(True)
        output = model(x_adv)

        # We want to minimize the output (push towards 0 = normal)
        loss = output
        loss.backward()

        with torch.no_grad():
            # Gradient step: move in direction that decreases output
            grad = x_adv.grad.sign()
            # Apply mask: only modify allowed features
            perturbation = -alpha * grad * mask_t
            x_adv = x_adv + perturbation

            # Project: clamp perturbation within epsilon ball
            delta = x_adv - x_orig_t
            delta = torch.clamp(delta, -epsilon, epsilon)
            # Re-apply mask to ensure non-modifiable features are unchanged
            delta = delta * mask_t
            x_adv = x_orig_t + delta

            # Clamp to valid feature ranges from training data
            x_adv = torch.max(x_adv, feat_min_t.unsqueeze(0))
            x_adv = torch.min(x_adv, feat_max_t.unsqueeze(0))

            x_adv = x_adv.detach()

    return x_adv.squeeze(0).cpu().numpy()

# --- 6. Run attacks for each epsilon ---
epsilons = [0.05, 0.1, 0.15, 0.2, 0.3]
checker = AttackPlausibilityChecker(feature_names)

print(f"\n{'='*70}")
print(f"PGD Evasion Attack Results")
print(f"{'='*70}")

for eps in epsilons:
    successful_attacks = 0
    plausible_attacks = 0
    total_attacks = len(tp_indices)

    for idx in tp_indices:
        x_orig = X_test[idx]
        orig_label = test_original_labels[idx]

        # Determine attack type for plausibility checking
        atk_type = attack_type_map.get(orig_label, 'DoS')

        # Run PGD with only HIGHLY_MODIFIABLE features first
        x_adv = pgd_attack(model, x_orig, highly_modifiable_mask,
                           epsilon=eps, alpha=0.01, num_iter=40,
                           feat_min=feature_min, feat_max=feature_max)

        # Check if attack succeeded
        x_adv_t = torch.tensor(x_adv, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            pred = model(x_adv_t).item()

        # If not successful with highly modifiable, try combined mask
        if pred >= 0.5:
            x_adv = pgd_attack(model, x_orig, combined_modifiable_mask,
                               epsilon=eps, alpha=0.01, num_iter=40,
                               feat_min=feature_min, feat_max=feature_max)
            x_adv_t = torch.tensor(x_adv, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():
                pred = model(x_adv_t).item()

        if pred < 0.5:
            successful_attacks += 1

            # Check plausibility
            is_plausible, violations = checker.check_plausibility(
                x_adv, attack_type=atk_type, scaler=scaler, verbose=False
            )
            if is_plausible:
                plausible_attacks += 1

    success_rate = successful_attacks / total_attacks * 100 if total_attacks > 0 else 0
    plausible_rate = plausible_attacks / successful_attacks * 100 if successful_attacks > 0 else 0

    print(f"\nEpsilon: {eps}")
    print(f"  Total attacks attempted:  {total_attacks}")
    print(f"  Successful attacks:       {successful_attacks} ({success_rate:.2f}%)")
    print(f"  Plausible attacks:        {plausible_attacks} ({plausible_rate:.2f}% of successful)")

Total features: 122
Highly modifiable: 4
Partially modifiable: 96
Combined modifiable: 100

Correctly classified attacks (true positives): 8903

PGD Evasion Attack Results

Epsilon: 0.05
  Total attacks attempted:  8903
  Successful attacks:       294 (3.30%)
  Plausible attacks:        19 (6.46% of successful)

Epsilon: 0.1
  Total attacks attempted:  8903
  Successful attacks:       515 (5.78%)
  Plausible attacks:        85 (16.50% of successful)

Epsilon: 0.15
  Total attacks attempted:  8903
  Successful attacks:       722 (8.11%)
  Plausible attacks:        111 (15.37% of successful)
